In [0]:
import data.utilities.common as Commonlib
from data.utilities.medallion import Medallion
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.functions import col, year
from pyspark.sql.types import IntegerType, TimestampType

In [0]:
config = Commonlib.get_object_config("./configuration/promo_plan_dtl.yml")

In [0]:
medallion = Medallion(config)

In [0]:
rm_deal_dtl = spark.read.table(MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_deal_dtl"))
rm_deal_hdr = (
    spark.read.table(MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_deal_hdr"))
    .drop("__created_tsp")
    .drop("last_update_date")
)
rm_nrp_vers = (
    spark.read.table(MTU.get_fully_qualified_table_name("oracle", "rmr", "rm_nrp_vers"))
    .drop("__created_tsp")
    .drop("last_update_date")
)

In [0]:
df = rm_deal_dtl.join(rm_deal_hdr, on="deal_hdr_seq_id")
df = df.join(rm_nrp_vers, on="nrp_vers_seq_id")

In [0]:
df = df[
    "cpm_promo_id",
    "deal_dtl_seq_id",
    "deal_hdr_seq_id",
    "curr_max_disc_amt",
    "qd_min_qty",
    "qd_max_qty",
    "plan_yr",
    "state_cd",
    "last_update_date",
    "__created_tsp",
]

In [0]:
df = (
    df.withColumnRenamed("cpm_promo_id", "cpm_promo_nbr")
    .withColumnRenamed("deal_dtl_seq_id", "promo_plan_dtl_id")
    .withColumnRenamed("deal_hdr_seq_id", "promo_plan_id")
    .withColumnRenamed("curr_max_disc_amt", "max_disc_amt")
    .withColumnRenamed("qd_min_qty", "qty_disc_min_qty")
    .withColumnRenamed("qd_max_qty", "qty_disc_max_qty")
    .withColumnRenamed("plan_yr", "plan_yr_nbr")
    .withColumnRenamed("state_cd", "sls_dest_cd")
    .withColumnRenamed("last_update_date", "edw_mod_tsp")
)

In [0]:
df = df.withColumn("edw_mod_tsp", col("edw_mod_tsp").cast(TimestampType())).withColumn(
    "plan_yr_nbr", year("plan_yr_nbr").cast(IntegerType())
)

In [0]:
medallion.df = df

In [0]:
try:
    # If the target table is empty, load the historic data first.
    if (spark.catalog.tableExists(MTU.get_fully_qualified_table_name("commercial", "promotions", "promo_plan_dtl"))) & (
        spark.read.table(MTU.get_fully_qualified_table_name("commercial", "promotions", "promo_plan_dtl")).count() > 0
    ):
        medallion.logger.info("Table already exists in the gold layer - skipping historic load.")
    else:
        historic_df = spark.read.table(
            MTU.get_fully_qualified_table_name("external", "static", "promo_plan_dtl_legacy")
        )

        # Lowercase columns
        for column in list(historic_df.columns):
            historic_df = historic_df.withColumnRenamed(column, column.lower())

        # Subset columns
        subset_columns = [{"name": x["name"], "type": x["type"]} for x in config["gold"]["schema"]]

        for column in subset_columns:
            if column["name"] not in historic_df.columns:
                historic_df = historic_df.withColumn(column["name"], lit(None).cast(column["type"]))

        historic_df = historic_df[[c["name"] for c in subset_columns]]

        historic_medallion = Medallion(config)
        historic_medallion.df = historic_df
        historic_medallion.write_to_delta(load_type="overwrite", layer="gold")

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
medallion.write_to_delta(load_type="upsert", layer="gold")

In [0]:
try:
    # publish to external stage - snowflake
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(config.get("catalog"), config.get("schema"), config.get("name"))
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

medallion.logger.info("publish activity completed successfully.")

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config.get("name", None),
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )